In [1]:
from sklearn.model_selection import cross_val_score, KFold, cross_validate, RandomizedSearchCV
from sklearn.feature_selection import VarianceThreshold, RFECV, RFE
from sklearn.preprocessing import PowerTransformer, RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline


from ml_enhance import plot_scaled_linreg_result, CorrelationFilter


import matplotlib.pyplot as plt
from sklearn import pipeline
import seaborn as sns
import pandas as pd
import numpy as np
import scipy
import json

In [2]:
def make_pipeline(model):
    return Pipeline([
        ("variance", VarianceThreshold(threshold=0.0)),
        ("remove_corr", CorrelationFilter(threshold=1.0)),
        ("transform", PowerTransformer(method='yeo-johnson', standardize=False)),
        ("scale", RobustScaler()),
        ("predict", model)
    ])

In [4]:
df = pd.read_csv("data/processed_dataset_wo_metals.csv")

df = df.drop("avg_atomic_quadrupole_principal_invariant_3", axis=1)

y = df["solubility"]
X = df.drop(["solubility", "smiles", "canon_smiles", "id"], axis=1)

In [13]:
rf_pipe = make_pipeline(RandomForestRegressor(random_state=42))

In [18]:
rf_param_dist = {
    "predict__n_estimators": [200, 500, 800],
    "predict__max_depth": [None, 10, 20, 30],
    "predict__max_features": ["sqrt", 0.5, 0.8],
    "predict__min_samples_split": [2, 5, 10],
    "predict__min_samples_leaf": [1, 2, 4]
}

rf_search = RandomizedSearchCV(
    rf_pipe,
    rf_param_dist,
    n_iter=30,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

In [19]:
rf_search.fit(X, y)

KeyboardInterrupt: 

In [ ]:
print("Best score:", rf_search.best_score_)
print("Best params:", rf_search.best_params_)